# 08 — Pathway-Level Analysis (R1, R3–R6, R8)

This notebook re-implements the pathway-level analyses from the original
`exploratory/08_pathway_analysis.ipynb` (R1–R8), corrected per the v2 review:

| ID | Analysis | Data |
|----|----------|------|
| R1 | AGORA2 enzyme evidence for SHAP producer species + Fisher enrichment test | AGORA2 GENRE reconstructions |
| ~~R2~~ | ~~KEGG metabolic-capacity proxy~~ | **dropped — see note below** |
| R3 | SCFA–polyamine producer enrichment (Fisher, BH-FDR; SCFA guild defined by AGORA2 EC numbers) | `ml_results_v2.pkl` (`pa_producers`), AGORA2 |
| R4 | Oxidative-stress metabolite index vs CRC stage / MOFA factors | `preprocessed_data_reduced_v2.pkl`, `mofa_factor_scores_v2.csv` |
| R5 | Oral-invasion index + dicarboxylic-acid correlations | `preprocessed_data_reduced_v2.pkl`, MOFA factors |
| R6 | Fusobacterium–polyamine correlations + AGORA2 evidence | `preprocessed_data_reduced_v2.pkl`, AGORA2 |
| ~~R7~~ | ~~eIF5A hypusination cascade mediation~~ | **dropped — see note below** |
| R8 | Polyamine / dicarboxylic-acid overlap (BH-FDR, always written) | `preprocessed_data_reduced_v2.pkl` |

## What changed vs the original notebook

1. **R1/R2/R6's genus-level KEGG organism-code lookups are replaced by
   AGORA2 species-level enzyme evidence.** The old approach inferred a
   *study* species' enzyme content from *any* organism sharing its genus in
   KEGG's organism list — a congener, not the species itself, and a live
   REST dependency with no local cache. We instead look the species up
   directly in its own AGORA2 genome-scale reconstruction via
   `utils.species_encodes_enzyme()`, which is `True`/`False` when AGORA2 has
   a model for that species and `None` (no evidence, not "no enzyme") when
   it doesn't.

2. **R2 (module-completeness check) is dropped entirely.** It only ever
   tested 5 pre-fetched enzyme KOs against an unrelated KEGG module's actual
   gene set and was provably non-functional — and it is now fully subsumed
   by the AGORA2 species-level check in R1/R6 above, which is strictly more
   specific evidence.

3. **R7 (eIF5A hypusination cascade mediation) is dropped entirely.** It
   depended on the `statsmodels.stats.mediation.Mediation` infrastructure
   that has been removed from this pipeline (see `07_mediation.ipynb`,
   which is now a short explanation of *why* mediation analysis was
   excluded — its producer-species mediator list came from the same-cohort
   SHAP ranking it then re-tested, and the `Mediation` API's 0/1-contrast
   assumption doesn't match an unscaled continuous exposure). There is no
   dangling call to any removed mediation code here.

4. **R1's producer-species labeling bug is fixed.** The old code built a
   combined producer list as `list(orn_prod['Species']) + list(put_prod['Species'])`
   and then labeled every entry `'Ornithine' if sp in list(orn_prod['Species'])
   else 'Putrescine'` — any species present in *both* lists (a genuine
   dual ornithine/putrescine producer) silently only ever got the
   `'Ornithine'` label, losing its putrescine-producer row entirely. Below,
   the label is attached at construction time so a dual-producer species
   correctly appears as two rows.

5. **R3 and R8 now apply BH-FDR correction** before calling anything
   significant. The old version used raw nominal p-values throughout,
   including a re-test of an already-selected "significant" species list
   (double dipping) with no correction anywhere; the double-dipped
   cross-cohort re-test is dropped rather than carried forward uncorrected.

6. **`r8_polyamine_dicarb_overlap_v2.csv` is always written**, even when the
   overlap table is empty — `09_pathway_network.ipynb` depends on this file
   existing unconditionally, and the old notebook only wrote it inside an
   `if not pad_df.empty:` guard.

7. **No local stage-map redefinition.** All stage encoding uses
   `utils.CRC_STAGE_ORDER` / `utils.STAGE_GROUP_MAP` / `utils.STAGE_GROUP_ORDER`
   (Healthy / Early_CRC / Advanced_CRC, with PostSurgery off the ordinal
   axis). MP is excluded from the study entirely.

9. **Only positively associated SHAP species count as candidate producers.**
   NB03 now stores a `Direction` per species (sign of the Spearman ρ between a
   species' abundance and its SHAP contribution); species whose higher abundance
   pushes the prediction towards *low* metabolite levels are not producers.

10. **R1 now includes an actual enrichment test** (Fisher's exact, one-sided, per
    polyamine, BH-corrected): are SHAP producers more likely to encode the
    biosynthetic enzyme than the other species with an AGORA2 model? The polyamine
    EC map (`utils.POLYAMINE_EC`) was also corrected: it previously listed enzymes
    that *consume* ornithine/arginine as their "producers".

11. **R3's SCFA guild is defined by AGORA2 EC numbers** (`utils.SCFA_EC`: butyrate
    2.8.3.8 / 2.7.2.7, propionate 2.8.3.1), not by a curated genus list — that list
    had been assembled from the same cohort's SHAP top genera, which made the
    enrichment circular.

12. **R4–R6 p-values are BH-corrected** within each table (`Q_*` columns).

8. **No live KEGG REST calls.** The old R1/R2/R6/R4/R5 made repeated,
   uncached `requests.get()` calls against `rest.kegg.jp` — both as (weak)
   enzyme evidence, now fully replaced by AGORA2, and as purely decorative
   "pathway context" printouts (e.g. listing KOs in map00910/map00071) that
   were never actually used by any test. Both uses are dropped, so this
   notebook has no network dependency and nothing to cache.

**MOFA factors are identified dynamically, not by hardcoded number.** The
old notebook hardcoded "Factor 5 = oxidative stress" and "Factor 3 = oral
invasion". Because NB06 now keeps every ARD-pruned/scree-justified factor
(not just a fixed top-3), the factor count and ordering can differ run to
run, so R4/R5 instead compute each candidate biological index directly from
metabolite/species abundances and then report which fitted factor
correlates with it most strongly — the index itself never depends on a
factor number.


In [ ]:
import sys
sys.path.insert(0, '.')

import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy.stats import spearmanr, kruskal, fisher_exact
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.multitest import multipletests

import utils
import viz

viz.apply_theme()   # one shared theme and font (DejaVu Sans, Type 42): see viz.py

RESULTS_DIR = utils.RESULTS_DIR
TABLES      = RESULTS_DIR / 'tables'
FIGURES     = RESULTS_DIR / 'figures' / 'pathway'
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

YACHIDA = 'YACHIDA-CRC-2019'

with open(RESULTS_DIR / 'intermediate' / 'preprocessed_data_reduced_v2.pkl', 'rb') as f:
    pdata = pickle.load(f)

with open(RESULTS_DIR / 'intermediate' / 'ml_results_v2.pkl', 'rb') as f:
    ml_results = pickle.load(f)

fac_scores   = pd.read_csv(TABLES / 'mofa_factor_scores_v2.csv')
fac_loadings = pd.read_csv(TABLES / 'mofa_factor_loadings_v2.csv')

spc_df  = pdata['transformed_species'][YACHIDA].copy()
mtb_df  = pdata['transformed_mtb'][YACHIDA].copy()
meta_df = pdata['harmonized_meta'][YACHIDA].copy()
polyamine_columns = pdata.get('polyamine_columns', {}).get(YACHIDA, {})   # {kegg_id: column_name}
mtb_kegg_map      = pdata.get('mtb_kegg_map', {}).get(YACHIDA, {})        # {column_name: kegg_id}

pa_producers_all = ml_results.get('pa_producers', pd.DataFrame(columns=['Species', 'Polyamine', 'Rank']))
# Candidate producers = species whose higher abundance pushes the model towards HIGH
# metabolite levels (NB03 'Direction'); negatively associated species are not producers.
if 'Direction' in pa_producers_all.columns:
    pa_producers = pa_producers_all[pa_producers_all['Direction'] == 'positive'].copy()
else:
    warnings.warn('pa_producers has no Direction column (old NB03 output) -- using all rows')
    pa_producers = pa_producers_all.copy()
print(f"Producer rows: {len(pa_producers_all)} total, {len(pa_producers)} positive-direction")

print(f"Loaded: spc={spc_df.shape}, mtb={mtb_df.shape}, meta={meta_df.shape}, "
      f"fac_scores={fac_scores.shape}, fac_loadings={fac_loadings.shape}")
print(f"pa_producers: {pa_producers.shape}, polyamines: "
      f"{sorted(pa_producers['Polyamine'].unique()) if not pa_producers.empty else '(empty)'}")

# ── Align every table to the same sample set (never assume matching row order) ──
common_samples = sorted(
    set(spc_df['Sample']) & set(mtb_df['Sample']) & set(meta_df['Sample']) & set(fac_scores['Sample'])
)
print(f"Common samples across spc/mtb/meta/factor-scores: {len(common_samples)}")

spc  = spc_df.set_index('Sample').loc[common_samples]
mtb  = mtb_df.set_index('Sample').loc[common_samples]
meta = meta_df.set_index('Sample').loc[common_samples]
fac  = fac_scores.set_index('Sample').loc[common_samples]

CRC_STAGE   = meta['CRC.Stage.Numeric'].astype(float)   # utils' corrected tiers
STAGE_GROUP = meta['Stage.Group']                        # Healthy/Adenoma/Early_CRC/Advanced_CRC
STUDY_GROUP = meta['Study.Group']


## Setup — column definitions & AGORA2 enzyme-evidence index

Metabolite columns are resolved by KEGG ID (via `mtb_kegg_map` / `polyamine_columns`
carried through from NB01/NB02, with a substring fallback on the
`"<KEGG_ID>_<Name>"` naming convention) rather than hardcoded literal column
strings — NB02's feature reduction to the top-variance metabolites only
protects polyamines, so dicarboxylic-acid / oxidative-stress marker columns
may or may not survive into `preprocessed_data_reduced_v2.pkl`; every section
below checks for and warns about missing columns rather than assuming they
exist.

The AGORA2 species→EC index is built once here and reused by R1 and R6.


In [ ]:
def find_columns_by_kegg(kegg_id_to_name, kegg_col_map, mtb_columns):
    """Resolve {kegg_id: column_name} using the column->kegg_id map carried
    through the pipeline, falling back to substring matching on the raw
    "<KEGG_ID>_<Name>" convention (see utils.find_polyamine_columns) for any
    id not covered by that map.
    """
    inv = {}
    for col, kid in (kegg_col_map or {}).items():
        inv.setdefault(kid, col)

    found = {}
    for kid, name in kegg_id_to_name.items():
        if kid in inv and inv[kid] in mtb_columns:
            found[kid] = inv[kid]
            continue
        for col in mtb_columns:
            if col.startswith(kid + '_'):
                found[kid] = col
                break
        else:
            matches = [c for c in mtb_columns if name.lower().replace('-', '') in c.lower().replace('-', '')]
            if matches:
                found[kid] = matches[0]
    return found


# ── Polyamines (prefer NB01's own polyamine_columns; fall back if incomplete) ──
PA_KEGG_TO_NAME = utils.POLYAMINE_KEGG  # {kegg_id: name}
pa_col_map = find_columns_by_kegg(PA_KEGG_TO_NAME, mtb_kegg_map, mtb.columns)
pa_col_map.update({k: v for k, v in polyamine_columns.items() if v in mtb.columns})
print("Polyamine columns resolved:", pa_col_map)
missing_pa = [k for k in PA_KEGG_TO_NAME if k not in pa_col_map]
if missing_pa:
    warnings.warn(f"Polyamine KEGG IDs not found in mtb columns (likely dropped by NB02's "
                   f"variance-reduction step): {missing_pa}")

# ── Dicarboxylic acids (Factor-3-associated FA beta-oxidation products) ──
DICARB_KEGG = {
    'C06104': 'Adipate',
    'C08277': 'Sebacate',
    'C02656': 'Pimelate',
    'C06103': '6-Hydroxyhexanoate',
}
dicarb_col_map = find_columns_by_kegg(DICARB_KEGG, mtb_kegg_map, mtb.columns)
print("Dicarboxylic-acid columns resolved:", dicarb_col_map)
if not dicarb_col_map:
    warnings.warn("No dicarboxylic-acid columns found — likely dropped by NB02's "
                   "top-variance metabolite reduction. R5/R8 dicarboxylic-acid "
                   "sections will report empty results.")

# ── Oxidative-stress markers (urate, glucuronate) ──
OXID_KEGG = {
    'C00366': 'Urate',
    'C00191': 'Glucuronate',
}
oxid_col_map = find_columns_by_kegg(OXID_KEGG, mtb_kegg_map, mtb.columns)
print("Oxidative-stress marker columns resolved:", oxid_col_map)
if not oxid_col_map:
    warnings.warn("No oxidative-stress marker columns (urate/glucuronate) found — "
                   "R4's oxidative-stress index will be unavailable.")

# ── Taxa of interest (name-based; no KEGG lookup needed) ──
ORAL_SPECIES = ['Haemophilus_D parainfluenzae', 'Veillonella atypica', 'Streptococcus salivarius']
ORAL_COLS    = [c for c in spc.columns if c in ORAL_SPECIES]
FUSO_COLS    = [c for c in spc.columns if 'Fusobact' in c]
BLAUTIA_COLS = [c for c in spc.columns if 'Blautia_A' in c or c == 'Mediterraneibacter torques']

print("\nOral species present:     ", ORAL_COLS)
print("Fusobacterium species:    ", FUSO_COLS)
print("Blautia/Mediterr species: ", BLAUTIA_COLS)


In [ ]:
# ── AGORA2 species-level enzyme evidence (replaces R1/R2/R6's genus-level KEGG
# organism-code lookups). The cache at RESULTS_DIR/tables/agora2_species_ec_index.json
# is compatible and loads in ~1s; force_rebuild is intentionally left False. ──
# Genomic evidence is UNAVAILABLE in this run: no genomic reference index is
# built (utils.AGORA2_ENABLED is False; a gutSMASH index is pending). Every
# query below returns None -- 'no evidence either way', never False.
agora2_ec_index = None
print("Genomic evidence: unavailable -- no genomic reference index is built; "
      "every genomic_evidence() query returns None.")


def genomic_evidence(species, kegg_id):
    """Tri-state genomic evidence: True / False / None (no evidence either way)."""
    return None  # unavailable, not negative


def bh_family(df, pcols):
    """BH-correct every p-value column in `pcols` JOINTLY (one family per table)
    and add a matching Q_<col> column for each. NaN p-values stay NaN."""
    pcols = [c for c in pcols if c in df.columns]
    if df.empty or not pcols:
        return df
    stacked = df[pcols].stack()                       # drops NaN
    if stacked.empty:
        return df
    q = pd.Series(multipletests(stacked.values, method='fdr_bh')[1], index=stacked.index)
    q = q.unstack()
    for c in pcols:
        df[f'Q_{c}'] = q[c].reindex(df.index) if c in q.columns else np.nan
    return df


n_agora_species = sum(genomic_evidence(s, 'C00134') is not None for s in spc.columns)
print(f"Reduced species with an AGORA2 model: {n_agora_species}/{spc.shape[1]}")


## R1 · AGORA2 Enzyme Evidence for Producer Species

For every (species, polyamine) pair in NB03's positive-direction SHAP producer list,
look up whether that species' own AGORA2 reconstruction encodes an enzyme that
**produces** that polyamine (`utils.species_encodes_enzyme`, EC map
`utils.POLYAMINE_EC`).

Then, per polyamine, a one-sided **Fisher's exact test**: among reduced species that have
an AGORA2 model, are the SHAP producers more likely to encode the enzyme than the
non-producers? BH-corrected across polyamines → `r1_agora2_enrichment_test_v2.csv`.
Species without an AGORA2 model carry no evidence either way and are excluded from the
test (not counted as negatives).

In [ ]:
r1_df = (pa_producers[['Species', 'Polyamine', 'Rank']]
         .drop_duplicates(subset=['Species', 'Polyamine'])
         .reset_index(drop=True))

kegg_by_polyamine_name = {v: k for k, v in utils.POLYAMINE_KEGG.items()}
r1_df['KEGG_ID'] = r1_df['Polyamine'].map(kegg_by_polyamine_name)
r1_df['Encodes_Enzyme'] = [
    genomic_evidence(sp, kid) if pd.notna(kid) else None
    for sp, kid in zip(r1_df['Species'], r1_df['KEGG_ID'])
]
r1_df['AGORA2_Model_Available'] = r1_df['Encodes_Enzyme'].notna()
r1_df = r1_df.sort_values(['Polyamine', 'Rank']).reset_index(drop=True)
r1_df.to_csv(TABLES / 'r1_agora2_producer_enrichment_v2.csv', index=False)

print(f"R1 — {len(r1_df)} (species, polyamine) producer rows evaluated against AGORA2.")
for pa_name in sorted(r1_df['Polyamine'].unique()):
    sub = r1_df[r1_df['Polyamine'] == pa_name]
    n_yes  = int((sub['Encodes_Enzyme'] == True).sum())
    n_no   = int((sub['Encodes_Enzyme'] == False).sum())
    n_none = int(sub['Encodes_Enzyme'].isna().sum())
    print(f"  {pa_name:12s}: n={len(sub):3d}  AGORA2-confirmed={n_yes:3d}  "
          f"AGORA2-refuted={n_no:3d}  no-AGORA2-model={n_none:3d}")
print("Saved: r1_agora2_producer_enrichment_v2.csv")

# ── Enrichment test per polyamine (universe = reduced species with an AGORA2 model) ──
enr_rows = []
for pa_name in sorted(r1_df['Polyamine'].unique()):
    kid = kegg_by_polyamine_name.get(pa_name)
    prod = set(r1_df.loc[r1_df['Polyamine'] == pa_name, 'Species'])
    ev = {s: genomic_evidence(s, kid) for s in spc.columns}
    univ = [s for s, e in ev.items() if e is not None]
    a = sum(1 for s in univ if s in prod and ev[s]);      b = sum(1 for s in univ if s in prod and not ev[s])
    c = sum(1 for s in univ if s not in prod and ev[s]);  d = sum(1 for s in univ if s not in prod and not ev[s])
    if a + b == 0:
        continue
    odds, p = fisher_exact([[a, b], [c, d]], alternative='greater')
    enr_rows.append({'Polyamine': pa_name, 'KEGG_ID': kid, 'n_universe': len(univ),
                     'Producers_encoding': a, 'Producers_not_encoding': b,
                     'Others_encoding': c, 'Others_not_encoding': d,
                     'Frac_producers_encoding': a / (a + b),
                     'Frac_others_encoding': c / (c + d) if (c + d) else np.nan,
                     'OddsRatio': odds, 'PValue': p})
# Explicit schema: with no testable polyamine (no species has genomic evidence) the
# frame had no columns, and to_csv wrote a header-less 2-byte file pandas cannot read.
R1_TEST_COLS = ['Polyamine', 'KEGG_ID', 'n_universe', 'Producers_encoding',
                'Producers_not_encoding', 'Others_encoding', 'Others_not_encoding',
                'Frac_producers_encoding', 'Frac_others_encoding', 'OddsRatio', 'PValue',
                'QValue']
r1_test = pd.DataFrame(enr_rows, columns=R1_TEST_COLS)   # QValue filled below
if not r1_test.empty:
    r1_test['QValue'] = multipletests(r1_test['PValue'], method='fdr_bh')[1]
r1_test.to_csv(TABLES / 'r1_agora2_enrichment_test_v2.csv', index=False)
print("\nR1 enrichment test (one-sided Fisher, BH across polyamines):")
print(r1_test.round(4).to_string(index=False) if not r1_test.empty else '  (no testable polyamines)')
print("Saved: r1_agora2_enrichment_test_v2.csv")


## R2 — dropped

The original R2 ("KEGG metabolic capacity proxy") tested five pre-fetched
enzyme KOs against an unrelated KEGG module's gene set and was provably
non-functional; it is fully subsumed by the AGORA2 species-level evidence in
R1 above, which is a direct, species-specific check rather than a genus-level
proxy. Not ported forward — no R2 cell follows.


## R3 · SCFA–Polyamine Producer Enrichment

Fisher's exact test for enrichment of short-chain-fatty-acid (SCFA) producers among the
positive-direction SHAP polyamine-producer species -- overall and per polyamine, BH-FDR
corrected as one family.

**SCFA producer = AGORA2 model encodes a butyrate (2.8.3.8, 2.7.2.7) or propionate
(2.8.3.1) synthesis enzyme** (`utils.SCFA_EC`, matched anywhere in the model via
`utils.species_has_any_ec`). The previous curated genus list had been assembled from the
same cohort's SHAP top genera, which made a "significant enrichment" partly circular. The
test universe is the reduced species that have an AGORA2 model (species without one carry
no evidence).

In [ ]:
SCFA_ECS = utils.SCFA_EC['butyrate'] + utils.SCFA_EC['propionate']

all_species      = list(spc.columns)
producer_species = set(pa_producers['Species'].unique()) if not pa_producers.empty else set()

scfa_ev = {s: None for s in all_species}  # no genomic reference index: SCFA status not callable
cls_df = pd.DataFrame({
    'Species':     all_species,
    'Is_SCFA':     [scfa_ev[s] for s in all_species],
    'Is_Producer': [s in producer_species for s in all_species],
})
cls_df = cls_df[cls_df['Is_SCFA'].notna()].copy()        # universe: species with an AGORA2 model
cls_df['Is_SCFA'] = cls_df['Is_SCFA'].astype(bool)
print(f"R3 universe: {len(cls_df)} reduced species with an AGORA2 model; "
      f"SCFA (EC-defined): {int(cls_df['Is_SCFA'].sum())}; producers in universe: "
      f"{int(cls_df['Is_Producer'].sum())}")

test_rows = []

ct_overall = pd.crosstab(cls_df['Is_SCFA'], cls_df['Is_Producer']).reindex(
    index=[False, True], columns=[False, True], fill_value=0
)
odds_all, p_all = fisher_exact(ct_overall.values)
test_rows.append({
    'Test': 'Overall_SCFA_vs_AnyProducer', 'OddsRatio': odds_all, 'PValue': p_all,
    'n_SCFA_producer':  int((cls_df['Is_SCFA'] & cls_df['Is_Producer']).sum()),
    'n_producer_total': int(cls_df['Is_Producer'].sum()),
    'n_SCFA_total':     int(cls_df['Is_SCFA'].sum()),
    'n_total':          len(cls_df),
})

univ = set(cls_df['Species'])
scfa_set = set(cls_df.loc[cls_df['Is_SCFA'], 'Species'])
for pa_name in (sorted(pa_producers['Polyamine'].unique()) if not pa_producers.empty else []):
    pa_set = set(pa_producers.loc[pa_producers['Polyamine'] == pa_name, 'Species']) & univ
    n_scfa_prod    = len(pa_set & scfa_set)
    n_nonscfa_prod = len(pa_set) - n_scfa_prod
    n_scfa_bg      = len(scfa_set) - n_scfa_prod
    n_nonscfa_bg   = len(univ) - len(scfa_set) - n_nonscfa_prod
    odds_pa, p_pa = fisher_exact([[n_scfa_prod, n_scfa_bg], [n_nonscfa_prod, n_nonscfa_bg]])
    test_rows.append({
        'Test': f'{pa_name}_SCFA_enrichment', 'OddsRatio': odds_pa, 'PValue': p_pa,
        'n_SCFA_producer': n_scfa_prod, 'n_producer_total': len(pa_set),
        'n_SCFA_total': len(scfa_set), 'n_total': len(univ),
    })

r3_df = pd.DataFrame(test_rows)
if not r3_df.empty:
    _, qvals, _, _ = multipletests(r3_df['PValue'], method='fdr_bh')
    r3_df['QValue']      = qvals
    r3_df['Significant'] = r3_df['QValue'] < utils.FDR_THRESHOLD
else:
    r3_df = pd.DataFrame(columns=['Test', 'OddsRatio', 'PValue', 'n_SCFA_producer',
                                   'n_producer_total', 'n_SCFA_total', 'n_total',
                                   'QValue', 'Significant'])

r3_df.to_csv(TABLES / 'r3_scfa_polyamine_enrichment_v2.csv', index=False)
print("R3 — SCFA (EC-defined) x polyamine-producer enrichment (BH-FDR corrected):")
print(r3_df.to_string(index=False))
print("Saved: r3_scfa_polyamine_enrichment_v2.csv")


## R4 · Oxidative-Stress Metabolite Index vs CRC Stage / MOFA Factors

Build an oxidative-stress index directly from urate + glucuronate abundance
(rather than assuming a fixed "Factor 5" identity), correlate it with CRC
stage, and report which fitted MOFA factor correlates with it most strongly
as a data-driven candidate "oxidative-stress axis" factor.


In [ ]:
oxid_cols_present = [oxid_col_map[k] for k in OXID_KEGG if k in oxid_col_map]

if oxid_cols_present:
    oxid_vals = mtb[oxid_cols_present]
    oxid_z = (oxid_vals - oxid_vals.mean()) / oxid_vals.std().replace(0, np.nan)
    oxidative_index = oxid_z.mean(axis=1)
else:
    oxidative_index = pd.Series(np.nan, index=mtb.index)
    warnings.warn('R4: no oxidative-stress marker columns available — index is all-NaN.')

if oxidative_index.notna().any():
    rho_oxid_stage, p_oxid_stage = spearmanr(oxidative_index, CRC_STAGE, nan_policy='omit')
else:
    rho_oxid_stage, p_oxid_stage = np.nan, np.nan
print(f"Oxidative-stress index x CRC stage: rho = {rho_oxid_stage:.3f}, p = {p_oxid_stage:.4f}"
      if pd.notna(rho_oxid_stage) else "Oxidative-stress index unavailable — skipping stage correlation.")

group_vals = [oxidative_index[STAGE_GROUP == g].dropna() for g in utils.STAGE_GROUP_ORDER]
group_vals = [g for g in group_vals if len(g) > 0]
if len(group_vals) > 1:
    kw_stat, kw_p = kruskal(*group_vals)
    print(f"Kruskal-Wallis across Stage.Group: H = {kw_stat:.2f}, p = {kw_p:.4f}")
else:
    kw_stat, kw_p = np.nan, np.nan

# ── Which fitted MOFA factor correlates most strongly with this index? ──
factor_cols = [c for c in fac.columns if c.startswith('Factor')]
factor_oxid_rows = []
for fcol in factor_cols:
    if oxidative_index.notna().any():
        rho_f, p_f = spearmanr(fac[fcol], oxidative_index, nan_policy='omit')
    else:
        rho_f, p_f = np.nan, np.nan
    factor_oxid_rows.append({'Factor': fcol, 'Rho_oxidative_index': rho_f, 'PValue': p_f})
factor_oxid_df = bh_family(pd.DataFrame(factor_oxid_rows), ['PValue'])
factor_oxid_df.to_csv(TABLES / 'r4_factor_oxidative_index_correlations_v2.csv', index=False)
if not factor_oxid_df.empty and factor_oxid_df['Rho_oxidative_index'].notna().any():
    factor_oxid_df = factor_oxid_df.reindex(
        factor_oxid_df['Rho_oxidative_index'].abs().sort_values(ascending=False).index
    )
    candidate_oxidative_factor = factor_oxid_df.iloc[0]['Factor']
    print(f"Candidate oxidative-stress-axis factor: {candidate_oxidative_factor} "
          f"(rho={factor_oxid_df.iloc[0]['Rho_oxidative_index']:.3f}, "
          f"BH q={factor_oxid_df.iloc[0]['Q_PValue']:.4f} across {len(factor_oxid_df)} factors)")
else:
    candidate_oxidative_factor = None

# ── Blautia / Mediterraneibacter x oxidative markers (direct correlation) ──
blautia_rows = []
for bsp in BLAUTIA_COLS:
    x = spc[bsp]
    rho_s, p_s = spearmanr(x, CRC_STAGE)
    row = {'Species': bsp, 'Rho_stage': rho_s, 'Pval_stage': p_s}
    for kid, name in OXID_KEGG.items():
        col = oxid_col_map.get(kid)
        if col is not None:
            rho, p = spearmanr(x, mtb[col])
        else:
            rho, p = np.nan, np.nan
        row[f'Rho_{name}'] = rho
        row[f'Pval_{name}'] = p
    blautia_rows.append(row)
blautia_df = pd.DataFrame(blautia_rows)
blautia_df = bh_family(blautia_df, [c for c in blautia_df.columns if c.startswith('Pval_')])
blautia_df.to_csv(TABLES / 'r4_blautia_oxidative_correlations_v2.csv', index=False)
print(f"\nSaved: r4_blautia_oxidative_correlations_v2.csv ({len(blautia_df)} species)")

# ── Figure ──
if oxidative_index.notna().any():
    plot_df = pd.DataFrame({'OxidativeIndex': oxidative_index.values, 'Stage': STUDY_GROUP.values})
    stage_order = [g for g in utils.CRC_STAGE_ORDER if g in plot_df['Stage'].unique()]
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.boxplot(data=plot_df, x='Stage', y='OxidativeIndex', order=stage_order, palette='viridis', ax=ax)
    ax.axhline(0, color='gray', lw=0.8, ls='--')
    ax.set_title(f"Oxidative-stress index by CRC stage\n(rho={rho_oxid_stage:.3f}, KW p={kw_p:.4f})")
    ax.set_xlabel('CRC Stage'); ax.set_ylabel('Oxidative-stress index (z-mean)')
    ax.tick_params(axis='x', rotation=25)
    plt.tight_layout()
    out_r4 = FIGURES / 'r4_oxidative_stress_index_by_stage_v2.png'
    fig.savefig(out_r4, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved: {out_r4}")
else:
    print("Skipping R4 figure — oxidative-stress index unavailable.")


## R5 · Oral-Invasion Index + Dicarboxylic-Acid Correlations

Compute an oral-invasion composite score from the three canonical
oral-invader species, correlate it with CRC stage (with an age/BMI-adjusted
partial correlation via OLS residualization), identify the fitted MOFA
factor it aligns with best, and correlate the individual
Factor-3-associated dicarboxylic acids with stage / oral invasion / that
factor.


In [ ]:
oral_invasion = spc[ORAL_COLS].mean(axis=1) if ORAL_COLS else pd.Series(np.nan, index=spc.index)

if ORAL_COLS:
    rho_oi_stage, p_oi_stage = spearmanr(oral_invasion, CRC_STAGE)
    print(f"Oral-invasion index x CRC stage: rho = {rho_oi_stage:.3f}, p = {p_oi_stage:.4f}")
else:
    rho_oi_stage, p_oi_stage = np.nan, np.nan
    warnings.warn('R5: none of the canonical oral-invader species were found — index is all-NaN.')

# ── Partial Spearman correlation controlling Age + BMI, via OLS residualization
# (avoids an undeclared pingouin dependency; equivalent to pg.partial_corr's
# Spearman variant for a single pair of continuous covariates). ──
df_part = pd.DataFrame({
    'oral_inv':  oral_invasion.values,
    'crc_stage': CRC_STAGE.values,
    'age':       meta['Age'].values,
    'bmi':       meta['BMI'].values,
}, index=meta.index).dropna()

if len(df_part) >= 10:
    X_cov = sm.add_constant(df_part[['age', 'bmi']])
    resid_oral  = df_part['oral_inv']  - sm.OLS(df_part['oral_inv'],  X_cov).fit().predict(X_cov)
    resid_stage = df_part['crc_stage'] - sm.OLS(df_part['crc_stage'], X_cov).fit().predict(X_cov)
    rho_partial, p_partial = spearmanr(resid_oral, resid_stage)
    print(f"Partial Spearman (adj. Age+BMI): rho = {rho_partial:.3f}, p = {p_partial:.4f} (n={len(df_part)})")
else:
    rho_partial, p_partial = np.nan, np.nan
    warnings.warn(f'R5: only {len(df_part)} complete Age/BMI/oral-invasion rows — skipping partial correlation.')

# ── Candidate oral-invasion-axis MOFA factor ──
factor_cols = [c for c in fac.columns if c.startswith('Factor')]
factor_oral_rows = []
for fcol in factor_cols:
    if oral_invasion.notna().any():
        rho_f, p_f = spearmanr(fac[fcol], oral_invasion, nan_policy='omit')
    else:
        rho_f, p_f = np.nan, np.nan
    factor_oral_rows.append({'Factor': fcol, 'Rho_oral_invasion': rho_f, 'PValue': p_f})
factor_oral_df = bh_family(pd.DataFrame(factor_oral_rows), ['PValue'])
factor_oral_df.to_csv(TABLES / 'r5_factor_oral_invasion_correlations_v2.csv', index=False)
if not factor_oral_df.empty and factor_oral_df['Rho_oral_invasion'].notna().any():
    factor_oral_df = factor_oral_df.reindex(
        factor_oral_df['Rho_oral_invasion'].abs().sort_values(ascending=False).index
    )
    candidate_oral_factor = factor_oral_df.iloc[0]['Factor']
    f_oral = fac[candidate_oral_factor]
    print(f"Candidate oral-invasion-axis factor: {candidate_oral_factor} "
          f"(rho={factor_oral_df.iloc[0]['Rho_oral_invasion']:.3f}, "
          f"BH q={factor_oral_df.iloc[0]['Q_PValue']:.4f} across {len(factor_oral_df)} factors)")
else:
    candidate_oral_factor = None
    f_oral = pd.Series(np.nan, index=fac.index)

# ── Dicarboxylic-acid correlations with stage / oral invasion / candidate factor ──
dc_results = []
for kid, name in DICARB_KEGG.items():
    col = dicarb_col_map.get(kid)
    if col is None:
        continue
    dc_vals = mtb[col]
    rho_s,  p_s  = spearmanr(dc_vals, CRC_STAGE)
    rho_oi, p_oi = spearmanr(dc_vals, oral_invasion) if oral_invasion.notna().any() else (np.nan, np.nan)
    rho_f,  p_f  = spearmanr(dc_vals, f_oral) if f_oral.notna().any() else (np.nan, np.nan)
    dc_results.append({
        'Metabolite': col, 'KEGG_ID': kid, 'Name': name,
        'Rho_stage': rho_s, 'P_stage': p_s,
        'Rho_OralInvasion': rho_oi, 'P_OralInvasion': p_oi,
        'Rho_CandidateOralFactor': rho_f, 'P_CandidateOralFactor': p_f,
    })
dc_df = pd.DataFrame(dc_results)
dc_df = bh_family(dc_df, ['P_stage', 'P_OralInvasion', 'P_CandidateOralFactor'])
dc_df.to_csv(TABLES / 'r5_dicarboxylic_acid_correlations_v2.csv', index=False)
print(f"\nSaved: r5_dicarboxylic_acid_correlations_v2.csv ({len(dc_df)} dicarboxylic acids)")
if dc_df.empty:
    warnings.warn('R5: no dicarboxylic-acid columns available — correlations table is empty.')

# ── Figure ──
if oral_invasion.notna().any():
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    stage_order = [g for g in utils.CRC_STAGE_ORDER if g in STUDY_GROUP.unique()]

    plot_oi = pd.DataFrame({'OralInvasion': oral_invasion.values, 'Stage': STUDY_GROUP.values})
    sns.boxplot(data=plot_oi, x='Stage', y='OralInvasion', order=stage_order, palette='magma_r', ax=axes[0])
    axes[0].set_title(f'Oral-invasion index by CRC stage\n(rho={rho_oi_stage:.3f}, p={p_oi_stage:.4f})')
    axes[0].tick_params(axis='x', rotation=30)

    if not dc_df.empty:
        first_dc_col = dc_df.iloc[0]['Metabolite']
        sc = axes[1].scatter(oral_invasion, mtb[first_dc_col], c=CRC_STAGE, cmap='RdYlBu_r', alpha=0.6, s=20)
        plt.colorbar(sc, ax=axes[1], label='CRC Stage (numeric)')
        axes[1].set_xlabel('Oral-invasion index'); axes[1].set_ylabel(first_dc_col)
        axes[1].set_title('Oral invasion vs dicarboxylic acid')
    else:
        axes[1].axis('off')
        axes[1].text(0.5, 0.5, 'No dicarboxylic-acid\ncolumns available', ha='center', va='center')

    plt.tight_layout()
    out_r5 = FIGURES / 'r5_oral_invasion_dicarb_v2.png'
    fig.savefig(out_r5, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved: {out_r5}")
else:
    print("Skipping R5 figure — oral-invasion index unavailable.")


## R6 · Fusobacterium–Polyamine Targeted Analysis

Direct Spearman correlations between each Fusobacterium species and
ornithine/putrescine/CRC stage, plus AGORA2 species-level evidence for
whether each Fusobacterium species actually encodes the terminal
ornithine/putrescine biosynthesis enzyme — replacing the old notebook's
hardcoded, unverified `FUSO_CODES = {'fnn', 'fpe', ...}` KEGG organism-code
list with the same species-level AGORA2 lookup used in R1.


In [ ]:
orn_col = pa_col_map.get('C00077')
put_col = pa_col_map.get('C00134')

fuso_rows = []
for fsp in FUSO_COLS:
    x = spc[fsp]
    rho_o, p_o = spearmanr(x, mtb[orn_col]) if orn_col is not None else (np.nan, np.nan)
    rho_p, p_p = spearmanr(x, mtb[put_col]) if put_col is not None else (np.nan, np.nan)
    rho_s, p_s = spearmanr(x, CRC_STAGE)

    fuso_rows.append({
        'Species': fsp,
        'Rho_ornithine': rho_o, 'Pval_ornithine': p_o,
        'Rho_putrescine': rho_p, 'Pval_putrescine': p_p,
        'Rho_stage': rho_s, 'Pval_stage': p_s,
        'AGORA2_Encodes_Ornithine_Enzyme':  genomic_evidence(fsp, 'C00077'),
        'AGORA2_Encodes_Putrescine_Enzyme': genomic_evidence(fsp, 'C00134'),
    })

fuso_df = pd.DataFrame(fuso_rows)
fuso_df = bh_family(fuso_df, ['Pval_ornithine', 'Pval_putrescine', 'Pval_stage'])
fuso_df.to_csv(TABLES / 'r6_fusobacterium_polyamine_correlations_v2.csv', index=False)

print(f"R6 — {len(fuso_df)} Fusobacterium species evaluated.")
if not fuso_df.empty:
    print(fuso_df.to_string(index=False))
else:
    warnings.warn('R6: no Fusobacterium species found in the reduced species table.')
print("Saved: r6_fusobacterium_polyamine_correlations_v2.csv")


## R7 — dropped

The original R7 ("eIF5A hypusination cascade") ran a putrescine ->
spermidine -> CRC-stage mediation analysis via
`statsmodels.stats.mediation.Mediation`. That infrastructure has been
removed from v2 (see `07_mediation.ipynb` for the full rationale: the
producer-species mediator list came from the same-cohort SHAP ranking it
then re-tested, and the `Mediation` API's 0/1-contrast assumption doesn't
match an unscaled continuous exposure). Not ported forward, and nothing
below references `Mediation`, `med_sig`, or any other removed mediation
symbol.


## R8 · Polyamine / Dicarboxylic-Acid Overlap

Compute a peroxisomal-beta-oxidation dicarboxylic-acid index (PC1 of the
available dicarboxylic acids), correlate it with CRC stage / oral invasion /
the candidate oral-invasion factor from R5, then test — with BH-FDR
correction applied separately within each family of tests (species x
polyamine; species x dicarboxylic acid) — which species correlate
significantly with *both* a polyamine and a dicarboxylic acid.
`r8_polyamine_dicarb_overlap_v2.csv` is written unconditionally, even when
no pair clears the threshold, since `09_pathway_network.ipynb` depends on
this file existing.


In [ ]:
dicarb_present_cols = [dicarb_col_map[k] for k in DICARB_KEGG if k in dicarb_col_map]

if len(dicarb_present_cols) >= 1:
    dc_data   = mtb[dicarb_present_cols].apply(lambda c: c.fillna(c.median()))
    dc_scaled = StandardScaler().fit_transform(dc_data)
    n_comp    = min(len(dicarb_present_cols), dc_scaled.shape[0])
    pca       = PCA(n_components=n_comp)
    dc_pca    = pca.fit_transform(dc_scaled)
    var_exp   = pca.explained_variance_ratio_

    print(f"Dicarboxylic-acid columns used: {dicarb_present_cols}")
    print(f"PC1 explains {var_exp[0]*100:.1f}% variance")
    print("PC1 loadings:", dict(zip(dicarb_present_cols, pca.components_[0].round(3))))

    dicarb_idx = dc_pca[:, 0]
    # Orient PC1 so that positive = more dicarboxylic acids (agrees with the mean
    # z-score of the acids). The old code flipped it based on its correlation with
    # stage, which made the 'positive = more dicarboxylic acids' label wrong.
    if np.corrcoef(dicarb_idx, dc_scaled.mean(axis=1))[0, 1] < 0:
        dicarb_idx = -dicarb_idx
    print("PC1 oriented: positive = more dicarboxylic acids "
          f"(r with mean z-score = {np.corrcoef(dicarb_idx, dc_scaled.mean(axis=1))[0, 1]:.2f}).")

    rho_di_stage, p_di_stage = spearmanr(dicarb_idx, CRC_STAGE)
    rho_di_oral,  p_di_oral  = (spearmanr(dicarb_idx, oral_invasion)
                                 if oral_invasion.notna().any() else (np.nan, np.nan))
    rho_di_factor, p_di_factor = (spearmanr(dicarb_idx, f_oral)
                                   if f_oral.notna().any() else (np.nan, np.nan))
    print(f"Dicarboxylic-acid index x CRC stage:              rho = {rho_di_stage:.3f}, p = {p_di_stage:.4f}")
    print(f"Dicarboxylic-acid index x oral invasion:          rho = {rho_di_oral:.3f}, p = {p_di_oral:.4f}")
    print(f"Dicarboxylic-acid index x candidate oral factor:  rho = {rho_di_factor:.3f}, p = {p_di_factor:.4f}")
else:
    dicarb_idx = np.full(len(mtb), np.nan)
    rho_di_stage = p_di_stage = rho_di_oral = p_di_oral = np.nan
    warnings.warn('R8: no dicarboxylic-acid columns available — index is all-NaN.')

# ── Species x polyamine correlations (one family of tests, BH-FDR corrected) ──
poly_rows = []
for sp in spc.columns:
    x = spc[sp]
    if x.std() == 0:
        continue
    for kid, col in pa_col_map.items():
        rho, p = spearmanr(x, mtb[col])
        if np.isnan(rho):
            continue
        poly_rows.append({
            'Species': sp, 'Polyamine': utils.POLYAMINE_KEGG.get(kid, kid), 'KEGG_ID': kid,
            'Rho_polyamine': rho, 'PValue_polyamine': p,
        })
poly_corr_df = pd.DataFrame(poly_rows)
if not poly_corr_df.empty:
    _, q, _, _ = multipletests(poly_corr_df['PValue_polyamine'], method='fdr_bh')
    poly_corr_df['QValue_polyamine'] = q

# ── Species x dicarboxylic-acid correlations (separate family, BH-FDR corrected) ──
dicarb_rows = []
for sp in spc.columns:
    x = spc[sp]
    if x.std() == 0:
        continue
    for kid, col in dicarb_col_map.items():
        rho, p = spearmanr(x, mtb[col])
        if np.isnan(rho):
            continue
        dicarb_rows.append({
            'Species': sp, 'Dicarb_Metabolite': col, 'Dicarb_KEGG_ID': kid,
            'Rho_dicarb': rho, 'PValue_dicarb': p,
        })
dicarb_corr_df = pd.DataFrame(dicarb_rows)
if not dicarb_corr_df.empty:
    _, q, _, _ = multipletests(dicarb_corr_df['PValue_dicarb'], method='fdr_bh')
    dicarb_corr_df['QValue_dicarb'] = q

Q_THRESH   = utils.FDR_THRESHOLD
RHO_THRESH = 0.15  # minimum-effect-size floor (matches original design's magnitude filter)

r8_cols = ['Species', 'Polyamine', 'KEGG_ID', 'Rho_polyamine', 'QValue_polyamine',
           'Dicarb_Metabolite', 'Dicarb_KEGG_ID', 'Rho_dicarb', 'QValue_dicarb']

if not poly_corr_df.empty and not dicarb_corr_df.empty:
    poly_sig = poly_corr_df[
        (poly_corr_df['QValue_polyamine'] < Q_THRESH) & (poly_corr_df['Rho_polyamine'].abs() > RHO_THRESH)
    ]
    dicarb_sig = dicarb_corr_df[
        (dicarb_corr_df['QValue_dicarb'] < Q_THRESH) & (dicarb_corr_df['Rho_dicarb'].abs() > RHO_THRESH)
    ]
    if not poly_sig.empty and not dicarb_sig.empty:
        r8_df = poly_sig.merge(dicarb_sig, on='Species', how='inner')[r8_cols]
    else:
        r8_df = pd.DataFrame(columns=r8_cols)
else:
    r8_df = pd.DataFrame(columns=r8_cols)
    warnings.warn('R8: species x polyamine or species x dicarboxylic-acid correlation '
                   'tables are empty (likely missing columns after NB02 reduction) — '
                   'overlap table will be empty.')

# --- Always write, even if empty -- NB09 depends on this file existing unconditionally. ---
r8_df.to_csv(TABLES / 'r8_polyamine_dicarb_overlap_v2.csv', index=False)
print(f"\nR8 overlap pairs found (QValue < {Q_THRESH}, |rho| > {RHO_THRESH} in both families): {len(r8_df)}")
if not r8_df.empty:
    print(r8_df.sort_values('QValue_dicarb').head(15).to_string(index=False))
print("Saved: r8_polyamine_dicarb_overlap_v2.csv (always written, per NB09 contract)")

# ── Figure ──
if not np.all(np.isnan(dicarb_idx)):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    stage_order = [g for g in utils.CRC_STAGE_ORDER if g in STUDY_GROUP.unique()]

    plot_di = pd.DataFrame({'DC_index': dicarb_idx, 'Stage': STUDY_GROUP.values})
    sns.boxplot(data=plot_di, x='Stage', y='DC_index', order=stage_order, palette='Blues_r', ax=axes[0])
    axes[0].axhline(0, color='gray', lw=0.8, ls='--')
    axes[0].set_title(f'Dicarboxylic-acid index (PC1) by CRC stage\nrho={rho_di_stage:.3f}, p={p_di_stage:.4f}')
    axes[0].tick_params(axis='x', rotation=30)

    if oral_invasion.notna().any():
        sc = axes[1].scatter(oral_invasion, dicarb_idx, c=CRC_STAGE, cmap='RdYlBu_r', alpha=0.6, s=20)
        plt.colorbar(sc, ax=axes[1], label='CRC Stage (numeric)')
        axes[1].set_xlabel('Oral-invasion index'); axes[1].set_ylabel('Dicarboxylic-acid index (PC1)')
        axes[1].set_title(f'Oral invasion vs dicarboxylic-acid index\nrho={rho_di_oral:.3f}, p={p_di_oral:.4f}')
    else:
        axes[1].axis('off')
        axes[1].text(0.5, 0.5, 'Oral-invasion index\nunavailable', ha='center', va='center')

    plt.tight_layout()
    out_r8 = FIGURES / 'r8_dicarboxylic_acid_index_v2.png'
    fig.savefig(out_r8, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved: {out_r8}")
else:
    print("Skipping R8 figure — dicarboxylic-acid index unavailable.")


## Summary — Pathway Analysis (R1, R3–R6, R8)

| ID | Analysis | Files written |
|----|----------|----------------|
| R1 | AGORA2 enzyme evidence for producer species + enrichment test | `r1_agora2_producer_enrichment_v2.csv`, `r1_agora2_enrichment_test_v2.csv` |
| ~~R2~~ | dropped (subsumed by R1) | — |
| R3 | SCFA (EC-defined)-polyamine producer enrichment (BH-FDR) | `r3_scfa_polyamine_enrichment_v2.csv` |
| R4 | Oxidative-stress index vs stage / factors (BH) | `r4_blautia_oxidative_correlations_v2.csv`, `r4_factor_oxidative_index_correlations_v2.csv`, `r4_oxidative_stress_index_by_stage_v2.png` |
| R5 | Oral-invasion index + dicarboxylic acids (BH) | `r5_dicarboxylic_acid_correlations_v2.csv`, `r5_factor_oral_invasion_correlations_v2.csv`, `r5_oral_invasion_dicarb_v2.png` |
| R6 | Fusobacterium-polyamine + AGORA2 evidence | `r6_fusobacterium_polyamine_correlations_v2.csv` |
| ~~R7~~ | dropped (mediation infrastructure removed) | — |
| R8 | Polyamine/dicarboxylic-acid overlap (BH-FDR, always written) | `r8_polyamine_dicarb_overlap_v2.csv`, `r8_dicarboxylic_acid_index_v2.png` |

**Required-by-contract outputs** (per `CONTRACTS.md`, consumed downstream by
`09_pathway_network.ipynb`): `r1_agora2_producer_enrichment_v2.csv`,
`r3_scfa_polyamine_enrichment_v2.csv`, and `r8_polyamine_dicarb_overlap_v2.csv`
(the last of these is always written, even when empty).
